# Data Processing

In [1]:
import polars as pl

import nwec.utils.excel
from nwec.constants import PROCESSED_UTILITY_DATA, RAW_UTILITY_DATA
from nwec.utils.cleaning import clean_utility_data, validate_data

In [2]:
spreadsheet = RAW_UTILITY_DATA / "IOU 200281 Data.xlsx"
sheet_name = "Arrearages"
sheet_index = nwec.utils.excel.get_sheet_index_from_name(spreadsheet, sheet_name)
core_value_column_name = "Arrearage_Amount"
processed_file_name = "arrearage_amounts"

df = pl.read_excel(spreadsheet, sheet_id=sheet_index, has_header=False)
df = df.slice(1)

In [3]:
# Copy month names to the next 3 columns for all 12 months
first_row = df.head(1)
second_row = df.slice(1, 1)
rest_rows = df.slice(2)

for month_idx in range(12):
    source_col_idx = 4 + (month_idx * 4)
    source_col_name = f"column_{source_col_idx + 1}"
    month_name = first_row[source_col_name][0]

    for offset in range(1, 4):
        target_col_name = f"column_{source_col_idx + offset + 1}"
        first_row = first_row.with_columns(pl.lit(month_name).alias(target_col_name))

    # Prepend month name to all 4 columns in the second row
    for offset in range(4):
        col_name = f"column_{source_col_idx + offset + 1}"
        second_row = second_row.with_columns((pl.lit(month_name + " ") + pl.col(col_name)).alias(col_name))

df = pl.concat([first_row, second_row, rest_rows])

In [4]:
df = df.slice(1)

new_column_names = df.row(0)
df = df.slice(1)
df.columns = new_column_names
id_cols = list(df.columns[:4])
value_cols = list(df.columns[4:])

df = df.unpivot(index=id_cols, on=value_cols, variable_name="month_vintage", value_name="value")

# Split the month_vintage column into Month and Vintage
df = df.with_columns(
    [
        pl.col("month_vintage").str.split(" ").list.first().alias("Month"),
        pl.col("month_vintage").str.split(" ").list.slice(1).list.join(" ").alias("Vintage"),
    ]
).drop("month_vintage")
df = df.select([*id_cols, "Month", "Vintage", "value"])

In [5]:
df = (
    df.filter(pl.col("Customer Class").str.contains(r"(?i)res"))
    .drop("Customer Class")
    .rename({"value": core_value_column_name})
)
df = clean_utility_data(df, value_column_name=core_value_column_name)
validate_data(df, value_column_name=core_value_column_name)

ValueError: Data validation failed:
  - Duplicates: Found 140 duplicate key combinations. Sample duplicates: [{'Utility': 'CNG', 'Year': 2022, 'Month': 10, 'Zip Code': '99352', 'Vintage': '60 Days', 'count': 2}, {'Utility': 'PAC', 'Year': 2022, 'Month': 1, 'Zip Code': '98942', 'Vintage': '30 Days', 'count': 3}, {'Utility': 'PAC', 'Year': 2022, 'Month': 1, 'Zip Code': '99362', 'Vintage': 'Total Arrearages', 'count': 3}, {'Utility': 'PAC', 'Year': 2022, 'Month': 1, 'Zip Code': '98953', 'Vintage': '30 Days', 'count': 3}, {'Utility': 'PAC', 'Year': 2022, 'Month': 1, 'Zip Code': '98901', 'Vintage': '90 Days +', 'count': 3}]. This may indicate overlapping data from different reporting periods.


In [ ]:
PROCESSED_UTILITY_DATA.mkdir(parents=True, exist_ok=True)
processed_path = PROCESSED_UTILITY_DATA / f"{processed_file_name}.arrow"

if processed_path.exists():
    combined_arrearage_amounts = pl.read_ipc(processed_path)
    combined_arrearage_amounts = pl.concat([combined_arrearage_amounts, df])
    combined_arrearage_amounts = combined_arrearage_amounts.unique()
else:
    combined_arrearage_amounts = df

# Sort by all columns to ensure consistent output order
combined_arrearage_amounts = combined_arrearage_amounts.sort(combined_arrearage_amounts.columns)

processed_path.unlink(missing_ok=True)
combined_arrearage_amounts.write_ipc(processed_path)
combined_arrearage_amounts.write_csv(processed_path.with_suffix(".csv"))